## Initial Imports

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

import os
import yaml
from crewai import Agent, Task, Crew, LLM

## Setup Model

We need to setup LLMs that our Agents can use. In this example, we use same Gemini Model for all Agents. However, we have the flexibility to use different tools for different Agents.

Refer [here](https://docs.crewai.com/concepts/llms#setting-up-your-llm) for details on setting up LLMs from other providers.

In [ ]:
MODEL='gemini/gemini-1.0'
GEMINI_API_KEY='API_KEY'

gemini_llm = LLM(
    model=MODEL,
    api_key=GEMINI_API_KEY,
    temperature=0,
)

## Loading Tasks and Agents YAML files

Load the `yaml` files that define agents and tasks. 

In [ ]:
# Define file paths for YAML configurations
files = {
    'agents': 'reddit_agents.yaml',
    'tasks': 'reddit_tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Create Custom Tool To Scrape Reddit 

This section showcases the creation of two custom tools, one for scraping a subreddit and second for searching a topic in subreddit.

In [ ]:
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field


class RedditScrapeInput(BaseModel):
  """Input schema for Reddit Scrape Tool."""
  subreddit_name: str = Field(..., description="Name of the subreddit to scrape.")
  limit: int = Field(default=33, description="Number of top posts to scrape.")


class RedditSearchInput(BaseModel):
  """Input schema for Reddit Scrape Tool."""
  subreddit_name: str = Field(..., description="Name of the subreddit to scrape.")
  topic_name: str = Field(..., description="Name of the topic to search.")
  limit: int = Field(default=33, description="Number of top posts to scrape.")
    
    
class RedditScrapeTool(BaseTool):
  name: str = "Reddit Scraper"
  description: str = (
      "Scrapes top posts and hot posts from a specified Reddit subreddit."
  )
  args_schema: Type[BaseModel] = RedditScrapeInput

  def _run(self, subreddit_name: str, limit:int) -> list:
    """
    Scrapes top Reddit posts and returns them as a list of dictionaries.

    Args:
      subreddit_name: The name of the subreddit to scrape.

    Returns:
      A list of dictionaries, where each dictionary represents a post 
      and contains the following keys:
        - 'title': The title of the post.
        - 'url': The URL of the post.
        - 'author': The username of the post author (if applicable).
        - 'score': The number of upvotes for the post.
        - 'created_utc': The Unix timestamp of when the post was created.
        - 'selftext': The self-text (body) of the post (if applicable). 
    """

    import praw
      
    client_id = 'client_id'
    client_secret = 'client_secret'
    user_agent = 'user_agent'

    reddit = praw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent
    )

    subreddit = reddit.subreddit(subreddit_name)

    top_posts = {}
    for submission in subreddit.top("week", limit=limit):
      post_data = {
          'title': submission.title,
          'url': submission.url,
          'author': submission.author.name if submission.author else None,
          'score': submission.score,
          'created_utc': submission.created_utc,
          'selftext': submission.selftext if submission.selftext else None
      }
      top_posts[submission.id] = post_data
    
    hot_posts = {}
    for submission in subreddit.hot(limit=limit):
      post_data = {
          'title': submission.title,
          'url': submission.url,
          'author': submission.author.name if submission.author else None,
          'score': submission.score,
          'created_utc': submission.created_utc,
          'selftext': submission.selftext if submission.selftext else None
      }
      hot_posts[submission.id] = post_data   
    
    merged_posts = {**top_posts, **hot_posts}  # hot_posts takes precedence
    values = list(merged_posts.values())
    return values

class RedditSearchTool(BaseTool):
  name: str = "Reddit Search"
  description: str = (
      "Scrapes top posts and hot posts from a specified Reddit subreddit."
  )
  args_schema: Type[BaseModel] = RedditSearchInput

  def _run(self, subreddit_name: str, topic_name:str, limit:int) -> list:
    """
    Search Reddit posts and returns them as a list.

    Args:
      subreddit_name: The name of the subreddit to scrape.
      topic_name: The name of the topic to search in subreddit.

    Returns:
      A list of dictionaries, where each dictionary represents a post 
      and contains the following keys:
        - 'title': The title of the post.
        - 'url': The URL of the post.
        - 'author': The username of the post author (if applicable).
        - 'score': The number of upvotes for the post.
        - 'created_utc': The Unix timestamp of when the post was created.
        - 'selftext': The self-text (body) of the post (if applicable). 
    """

    import praw
      
    client_id = 'client_id'
    client_secret = 'client_secret'
    user_agent = 'user_agent'

    reddit = praw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent
    )
    results = []
    subreddit = reddit.subreddit(subreddit_name)

    for submission in subreddit.search(topic_name, limit=limit):
        results.append({
            "title": submission.title,
            "url": submission.url,
            "score": submission.score,
            "num_comments": submission.num_comments,
            "created_utc": submission.created_utc,
            "id": submission.id
        })

    return results

## Create Agents

This segment demonstrates how to create Agents for specific roles.

In [ ]:
# Creating Agents
scrapper_agent = Agent(
  config=agents_config['scrapper'],
  llm=gemini_llm,
  allow_delegation=False,
  tools=[RedditScrapeTool()]
)

search_agent = Agent(
  config=agents_config['search_agent'],
  llm=gemini_llm,
  allow_delegation=False,
  memory=True,
  tools=[RedditSearchTool()]
)

analysis_agent = Agent(
  config=agents_config['reporting_analyst'],
  allow_delegation=False,
  llm=gemini_llm
)

## Create tasks
Here’s how to define specific tasks for your Agents.

In [ ]:
# Creating Tasks
report_data_collection = Task(
  config=tasks_config['scrape_task'],
  agent=scrapper_agent
)

report_generation = Task(
  config=tasks_config['reporting_task'],
  context=[report_data_collection],
  agent=analysis_agent,
)

search_data = Task(
  config=tasks_config['search_task'],
  agent=search_agent
)

search_report_generation = Task(
  config=tasks_config['reporting_task'],
  context=[search_data],
  agent=analysis_agent,
)

## Create Crews

This segment demonstrates how to assemble Agents and Tasks into a Crew to achieve a project goal.

In [ ]:
# Creating Crew
scrape_crew = Crew(
  agents=[
    scrapper_agent,
    analysis_agent
  ],
  tasks=[
    report_data_collection,
    report_generation
  ],
  verbose=True
)

search_crew = Crew(
  agents=[
    search_agent,
    analysis_agent
  ],
  tasks=[
    search_data,
    search_report_generation
  ],
  verbose=True
)

## Kickoff Crew

In [ ]:
inputs = {
    'subreddit_name':"....",
    'limit':50
}
scrape_result = scrape_crew.kickoff(inputs=inputs)

In [ ]:
inputs = {
    'subreddit_name':"....",
    'topic_name':"topic",
    'limit':50
}
search_result = search_crew.kickoff(inputs=inputs)

## Report

In [ ]:
from IPython.display import Markdown
scrape_markdown  = scrape_result.raw

In [ ]:
Markdown(scrape_markdown)

In [ ]:
from IPython.display import Markdown
search_markdown  = search_result.raw

In [ ]:
Markdown(search_markdown)